# 🚀 Kiara Foundation Model — Cloud GPU Pre-Training (Google Colab / Kaggle)

This notebook trains the **Kiara-Small (~125M parameter)** foundation model on free cloud GPUs (NVIDIA T4 or A100) using **PyTorch Mixed Precision (AMP)**.

### Why train here?
- **50x Faster**: ~15–20 minutes on a free T4 GPU vs 15+ hours on a laptop CPU.
- **Zero Laptop Wear**: Your laptop stays cool, safe, and silent.

## 1. Verify Cloud GPU Acceleration

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Warning: GPU is not active. Go to Runtime -> Change runtime type -> Select T4 GPU.")

## 2. Mount Google Drive (To Persist Trained Checkpoints)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
CHECKPOINT_BACKUP_DIR = '/content/drive/MyDrive/kiara_checkpoints'
os.makedirs(CHECKPOINT_BACKUP_DIR, exist_ok=True)
print(f"Checkpoints will be automatically backed up to: {CHECKPOINT_BACKUP_DIR}")

## 3. Clone Kiara Repository & Install Dependencies

In [ ]:
# Clone the repository (or pull latest updates)
%cd /content
!git clone https://github.com/varunvj151/kiara.git || (cd kiara && git pull)
%cd /content/kiara

# Install dependencies
!pip install -q -r requirements.txt

## 4. Run Pre-Flight Smoke Test (Verifies GPU Tensors & Backprop in < 5 seconds)

In [ ]:
!python experiments/smoke_test.py

## 5. Launch Stage 26D.1 Pre-Training (Kiara-Small 125M)

In [ ]:
# Execute Stage 26D.1 training
!python experiments/pretrain_26D1.py

# Backup checkpoints to Google Drive
!cp -r checkpoints/stage_26D1/* /content/drive/MyDrive/kiara_checkpoints/
print("✅ Training complete! Checkpoints safely saved in Google Drive.")

## 6. Test Model Generation

In [ ]:
import torch
from kiara.model import KiaraConfig, KiaraForCausalLM
from kiara.tokenizer import KiaraTokenizer
from kiara.trainer import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = KiaraTokenizer.load()
config = KiaraConfig.from_pretrained("checkpoints/stage_26D1")
model = KiaraForCausalLM(config).to(device)

load_checkpoint("checkpoints/stage_26D1/checkpoint_final.pt", model=model)
model.eval()

prompt = "The key mechanism in transformer architectures"
input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)

with torch.no_grad():
    for _ in range(30):
        logits = model(input_ids).logits[:, -1, :]
        next_tok = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_tok], dim=1)

output_text = tokenizer.decode(input_ids[0].tolist())
print("\n--- Model Output ---")
print(output_text)